In [ ]:
import sys
import copy
import random
import hashlib
import itertools
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline
import operator
import json
import pandas as pd
import numpy as np
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, average_precision_score
from sklearn.model_selection import validation_curve, learning_curve
from IPython.display import Image
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

يمكن تنزيل البيانات على 
https://drive.google.com/open?id=1H7N3Y7PEm0_442koQeffVfodlOiR0W_o
https://drive.google.com/open?id=1YbI2DFpuR_689OcqTwvRvEgcpu6BXSVt


In [ ]:
df = pd.read_csv('sites_markup.csv')

In [ ]:
df.head(3)


## الجزء الأول. وصف مجموعة البيانات والميزات



تم جمع مجموعة البيانات من مواقع البيع بالتجزئة للبقالة.<br>
جميعها لها بنية مماثلة، لذا من المفترض أن يكون من الممكن تحليل هذه المواقع ليس عن طريق القواعد اليدوية، ولكن عن طريق الزاحف الذي يمكنه التمييز بين العناصر الموجودة في الموقع.<br>
دعونا نلقي نظرة على المواقع


In [ ]:
Image(url='https://habrastorage.org/webt/xe/ek/vu/xeekvu2aogn6mljs-obb4y7nns0.png', width=700)

In [ ]:
Image(url='https://habrastorage.org/webt/vz/vz/u8/vzvzu80khqbtlr3cow9xmj8kgwq.png', width=700)

In [ ]:
Image(url='https://habrastorage.org/webt/ng/s3/8w/ngs38wzj4gtl4bnhikmamyjtory.png', width=700)


لذا فهما متشابهان بالفعل.<br>
دعونا نلقي نظرة على مجموعة البيانات والميزات


In [ ]:
df.info()

In [ ]:
df.head(2)


تم إنشاء الميزات من المنطق السليم والأوراق ذات الهدف المماثل (https://medium.com/contentsquare-engineering-blog/automatic-zone-recognition-in-webpages-68fb2efab822، https://arxiv.org/pdf/1210.6113.pdf).<i>childs_tags</i> - العلامات التي تحتوي على عناصر متداخلة<br>
<i>عمق</i> - العمق في شجرة ترميز HTML<br>
<i>element_classes_count</i> - كم عدد الفئات التي تحتوي على عنصر<br>
<i>element_classes_words</i> - أسماء فئات العناصر<br>
<i>href_seen</i> - يوجد href في مكان ما داخل العناصر الفرعية<br>
<i>img_seen</i> - موجود في مكان ما داخل العناصر الفرعية<br>
<i>inner_elements</i> - كم عدد العناصر المتداخلة<br>
<i>is_displayed</i> - علامة is_displayed من السيلينيوم<br>
<i>is_href</i> - هو العنصر الحالي href<br>
<i>location</i> - موقع الزاوية العلوية اليسرى في الصفحة<br>
<i>parent_tag</i> - أي علامة لها أصل العنصر<br>
<i>screenshot</i> - المسار إلى لقطة شاشة العنصر<br>
<i>shop</i> - اسم المتجر<br>
<i>siblings_count</i> - كم عدد العناصر الأخرى الموجودة في نفس الطبقة<br>
<i>size</i> - عرض العنصر وارتفاعه<br>
<i>tag</i> - علامة العنصر<br>
<i>text</i> - نص داخل العنصر<br>
<i>y</i> - هل هي "بطاقة منتج" هدفنا<br>



هدفنا هو العثور على كائن "بطاقة المنتج".


In [ ]:
Image(url='https://habrastorage.org/webt/if/yo/r2/ifyor2xutwypu-eyqfufqnh2g5o.png', width=900)


كما نرى أن هناك العديد من المنتجات في نفس الصفحة. تحتوي كل بطاقة منتج على العديد من العناصر التنازلية وبنية محددة مثل الصورة والسعر واسم المنتج والتقييم وعربة التسوق وما إلى ذلك.لم يتم جمع البيانات عن طريق تنزيل الصفحات للحصول على لغة HTML الأولية فحسب، بل عن طريق تنفيذ الصفحة للحصول على أحجام العناصر ومواقعها.
عندما نحصل على التعرف التلقائي على بطاقة المنتج، ستتمكن برامج الزحف لدينا من الحصول على المنتجات من مجموعة واسعة من المواقع دون قواعد صريحة. من الممكن الزحف إلى جميع المواقع يدويًا، ولكن بعد أكثر من 20 موقعًا، يكون من الصعب صيانتها، لأن بنية الموقع تتغير بمرور الوقت.



### الجزء 1.5 معالجة البيانات الأولية



يجب تصحيح "علامات_الأطفال"، و"الموقع"، و"علامة_الوالدين"، و"الحجم". ستكون جميع عناصر المصفوفة عبارة عن سلاسل ذات فاصل " "، وستكون جميع العناصر int عبارة عن حقول منفصلة.


In [ ]:
df[['loc_x', 'loc_y']] = df['location'].str.replace("'", '"').apply(json.loads).apply(pd.Series)
df[['size_h', 'size_w']] = df['size'].str.replace("'", '"').apply(json.loads).apply(pd.Series)
df.size_h = df.size_h.astype(int)
df.size_w = df.size_w.astype(int)

In [ ]:
df['parent_tag'] = df.parent_tag.str.split('_')
df['childs_tags'] = df.childs_tags.apply(lambda x: x.replace('[', '').replace(']','').strip())


إسقاط بعض الأعمدة


In [ ]:
df_clean = df.copy()
df_clean.drop(['location', 'size', 'Unnamed: 0'], axis=1, inplace=True)


## الجزء 2-3-4. تحليل البيانات الاستكشافية والتحليل البصري للميزات. الأنماط والرؤى وخصائص البيانات



#### التوزيع المستهدف
أول الأشياء أولا. دعنا نجد توزيع الميزة المستهدفة.


In [ ]:
df_clean['y'].value_counts()


3 آلاف قيمة حقيقية فقط، أي توزيع 42:1. ليس من المستغرب أن نحصل على كل عنصر من عناصر شجرة DOM كصف يبدأ من النص.



لقد قمت بعمل لقطة شاشة لكل عنصر، ليس من أجل التعلم العميق والتعرف على الصور ولكن لتسهيل فهم كيف يبدو العنصر. (لقطة الشاشة لكل عنصر تحتوي على عدد كبير جدًا من الصور، لذلك سأقوم بعمل لقطة شاشة فوق الرسم البياني باستخدام لقطات الشاشة =))
كيف تبدو العناصر المستهدفة:


In [ ]:
Image(url='https://habrastorage.org/webt/bz/xy/ds/bzxydsgez9atzsqavsatzgibrlm.png', width=900)


في الواقع مشابهة جدا



#### احصائيات عامة


In [ ]:
df_clean.head(1)

In [ ]:
continuous_vars = ['depth', 'element_classes_count', 'inner_elements', 'siblings_count', 'loc_x', 'loc_y', 'size_h', 'size_w']

In [ ]:
df_clean[continuous_vars].mean()

In [ ]:
df_clean[df_clean.y == 1][continuous_vars].mean()


#### العمق
في أي عمق يجلس العناصر المستهدفة


In [ ]:
df_clean[df_clean.y == 1].groupby('shop')['depth'].mean()


سلوك غريب لـ'Окей'. دعونا التحقيق.


In [ ]:
df_clean[(df_clean.shop ==  'Окей') & (df_clean.y == 1) & (df_clean.depth == 13)].shape

In [ ]:
df_clean[(df_clean.shop ==  'Окей') & (df_clean.y == 1) & (df_clean.depth != 13)].shape


جميع بطاقات المنتجات تقريبًا من عمق 'Окей' == 13، دعنا نلقي نظرة عليها
صور لـ df_clean[(df_clean.shop == 'Окей') & (df_clean.y == 1) & (df_clean.عمق == 13)]


In [ ]:
Image(url='https://habrastorage.org/webt/vf/uz/rd/vfuzrdfzhrzihvarhglnfk4zdje.png', width=900)

صور لـ df_clean[(df_clean.shop == 'Окей') & (df_clean.y == 1) & (df_clean.عمق != 13)]


In [ ]:
Image(url='https://habrastorage.org/webt/32/ee/ah/32eeahfoleo-ijciyzz1dvbv8hs.png', width=900)


يبدو نفسه


In [ ]:
df_clean[(df_clean.shop ==  'Окей') & (df_clean.y == 1) & (df_clean.depth == 13)].head(1)

In [ ]:
df_clean[(df_clean.shop ==  'Окей') & (df_clean.y == 1) & (df_clean.depth != 13)].head(1)


نفس العلامات وأسماء الفئات، تختلف العناصر الداخلية قليلاً. كل شيء صحيح، فقط العمق يختلف


In [ ]:
sns.barplot(df_clean.depth, df_clean.y)


عمق الطبقة الخارجية ==16


In [ ]:
df_clean[(df_clean.y == 1) & (df_clean.depth == 16)].head(1)

In [ ]:
Image(url='https://habrastorage.org/webt/gm/fz/hi/gmfzhiial3a_fxgj5lfq2oaa-ac.png', width=900)


يبلغ عمق معظم البطاقات في "Пекрасток" 9


In [ ]:
df_clean[(df_clean.y == 1) & (df_clean.shop == 'Перекрёсток')].head(1)

In [ ]:
Image(url='https://habrastorage.org/webt/na/us/e4/nause4he1lpfur_l-hvvjug12tk.png', width=900)


توجد بطاقات منتجات مختلفة في متجر واحد في حقيبة زاحفة أو يحتوي المتجر فقط على بطاقات مختلفة. في نوع واحد لا يوجد أي تصنيف أو to_cart



#### أحجام العناصر


In [ ]:
df_clean[df_clean.y == 1].groupby('shop')[['size_w', 'size_h']].mean()


"كوموس" يختلف كثيرًا
دعونا نرسم جميع أحجام بطاقة المنتج


In [ ]:
tmp_df = df_clean[(df_clean.y == 1)]
sns.scatterplot(x=tmp_df.size_w, y=tmp_df.size_h)

In [ ]:
tmp_df = df_clean[(df_clean.y == 1)]
g = sns.FacetGrid(tmp_df, col='shop', hue='y')
g.map(sns.scatterplot, 'size_w', 'size_h')


أولاً، توقعنا مقاسًا واحدًا لمتجر واحد ولكن هذا ليس صحيحًا. بالنسبة لحجم "Precrestok" و"Commus" يختلف كثيرًا. عندما نرسم جميع أحجام البطاقات، نرى أنها مخصصة في الغالب في الزاوية العلوية اليسرى من قطعة الأرض، بأحجام تبلغ حوالي 400 × 220. ولكن ما هو الخطأ في "كوموس"؟ دعونا زيارة الموقع.


In [ ]:
Image(url='https://habrastorage.org/webt/ua/pg/l6/uapgl6ei_c1knebnz9ls8rvlbdq.png', width=900)


الهيكل مختلف. في معظم المحلات التجارية، نرى تخطيطًا للبلاط، ولكن هنا تخطيط القائمة. من أجل البساطة، سأستبعد "COMUS" من إطار البيانات الخارجي.


In [ ]:
print(f'unique shops - {df_clean.shop.unique()}')
df_clean = df_clean[df_clean.shop != 'Комус']
print(f'unique shops after drop - {df_clean.shop.unique()}')


الآن دعونا نرى كيف تتوافق الأحجام المستهدفة مع جميع الأحجام الأخرى


In [ ]:
tmp_df = df_clean
g = sns.FacetGrid(tmp_df, col='shop', hue='y')
g.map(sns.scatterplot, 'size_w', 'size_h')


القيم المتطرفة تشوش الصورة


In [ ]:
tmp_df = df_clean[(df_clean.size_w < 1000) & (df_clean.size_h < 4000)]
g = sns.FacetGrid(tmp_df, col='shop', hue='y')
g.map(sns.scatterplot, 'size_w', 'size_h')


تكبير أقرب


In [ ]:
tmp_df = df_clean[(df_clean.size_w > 200) &(df_clean.size_w < 300) & (df_clean.size_h < 600)]
g = sns.FacetGrid(tmp_df, col='shop', hue='y')
g.map(sns.scatterplot, 'size_w', 'size_h')


تبدو ميزة رائعة ذات ارتباط قوي بين المتاجر والتحسينات من عناصر DOM الأخرى



#### الموقع


In [ ]:
tmp_df = df_clean
g = sns.FacetGrid(tmp_df, col='shop', hue='y')
g.map(sns.scatterplot, 'loc_x', 'loc_y' )


الموقع في المواقف السلبية؟!


In [ ]:
tmp_df = df_clean[(df_clean.loc_x > 0) & (df_clean.loc_y > 0)]
g = sns.FacetGrid(tmp_df, col='shop', hue='y')
g.map(sns.scatterplot, 'loc_x', 'loc_y' )


المواقف كما توقعنا. بنية جدولية قوية، ربما يمكننا تقديم ميزة جديدة من هذه الحقيقة.
القيم المتطرفة في 'Пекрасток'، دعونا نرى


In [ ]:
df_clean[(df_clean.loc_x > 900) & (df_clean.y == 1) & (df_clean.shop == 'Перекрёсток')].head(2)

In [ ]:
Image(url='https://habrastorage.org/webt/hz/oe/e5/hzoee5akuroyw3hedikww33qg1k.png', width=900)


؟؟؟؟؟
لا أعرف ما هو. إسقاط أفضل.


In [ ]:
print(f'rows in - {df_clean.shape[0]}')
df_clean = df_clean.drop(df_clean[(df_clean.loc_x > 900) & (df_clean.y == 1) & (df_clean.shop == 'Перекрёсток')].index)
print(f'rows after drop - {df_clean.shape[0]}')


#### معروض


In [ ]:
df_clean.groupby('is_displayed')['y'].sum()


جميع الصفوف is_displayed = true - ميزة زائدة عن الحاجة


In [ ]:
df_clean.drop(labels='is_displayed', axis=1, inplace=True)


#### العناصر_الداخلية


In [ ]:
df_clean['inner_elements'].mean()

In [ ]:
df_clean[df_clean.y == 1]['inner_elements'].mean()

In [ ]:
sns.distplot(df_clean.inner_elements);

In [ ]:
sns.distplot(df_clean[(df_clean.inner_elements > 2) & (df_clean.inner_elements < 100) ].inner_elements);

ربما كانت الميزة سيئة التصميم. بالنسبة لكل عنصر، نحسب العنصر الداخلي حتى الأوراق، لذلك في "الجسم" سيتم التعامل مع جميع العناصر الأخرى على أنها داخلية. 



#### العلامة


In [ ]:
df_clean['tag'].unique()

In [ ]:
df_clean[df_clean.y == 1]['tag'].unique()


بطاقاتنا هي عناصر "div" فقط. كم عدد شعبة أخرى؟


In [ ]:
df_clean[df_clean.tag == 'div'].groupby('y').size()

In [ ]:
df_clean.groupby('tag').size()


ومع ذلك، هناك الكثير من عناصر div التي ليست هدفنا، ولكن الميزة جيدة.



#### عدد الأشقاء


In [ ]:
df_clean[df_clean.y == 1]['siblings_count'].unique()

In [ ]:
df_clean[df_clean.y == 1].groupby('shop')['siblings_count'].agg(['unique'])


تم تصميم هذه الميزة لإظهار عدد "الأشقاء" حول عنصر ما. من المفترض أن توضع بطاقات المنتجات في قوائم بحيث تكون على نفس المستوى. إذا كان 30 منتجًا في كل صفحة، فمن المفترض أن يكون عدد الأشقاء 29. ولكن هناك خطأ ما. دعونا تحقق من المواقع الأصلية. على سبيل المثال.


In [ ]:
Image(url='https://habrastorage.org/webt/it/gk/7t/itgk7tfxsnyzsnda-zmiejtnu5o.png', width=900)

In [ ]:
Image(url='https://habrastorage.org/webt/em/i-/v9/emi-v9grsih0_i5rhbpnmhuloec.png', width=900)

In [ ]:
Image(url='https://habrastorage.org/webt/vr/in/0y/vrin0yedugz7xudlg7bpmhm8oi0.png', width=900)


نرى أن هناك قائمة "ul" تحتوي على "li" تحتوي على "div". لذا فإن العنصر "li" سيتصرف كما نتوقع - الأشقاء الصحيحون، ولكن "div" داخل الأشقاء "li" == 1. من المفترض أن تتم إعادة صياغته أو ربما تكون الميزة المثمرة ميتة.



#### is_href


In [ ]:
df_clean[df_clean.y == 1]['is_href'].mean()

In [ ]:
df_clean[df_clean.y == 0]['is_href'].mean()


بطاقات منتجاتنا غير قابلة للنقر



#### element_classes_words


In [ ]:
for shop, tmp_df in df_clean[df_clean.y == 1].groupby('shop'):
    print(shop)
    print(tmp_df['element_classes_words'].unique())



إنها تسمية فئة CSS التي تعتمد كليًا على المطورين. لكن في بعض الحالات، نرى أسماء مثل "منتج"، "سلعة"، قد تكون مفيدة. لا توجد بيانات كافية للقول على وجه اليقين.



####parent_tag


In [ ]:
for shop, tmp_df in df_clean[df_clean.y == 1].groupby('shop'):
    print(shop)
    print(tmp_df['parent_tag'].head(1))
               


#### علامات_الأطفال


In [ ]:
for shop, tmp_df in df_clean[df_clean.y == 1].groupby('shop'):
    print(shop)
    print(tmp_df['childs_tags'].head(1))
               


إنه ترميز بسيط ومحاكٍ بين الوالدين والطفل لجميع المتاجر باستثناء "Пеклёсток". ولكن يجب أن نكون مستعدين لهذا. في هذه الحالة لم نتمكن من استخدام هذه الميزات.



#### نص


In [ ]:
df_clean[df_clean.text.notna()]

ميزة "النص" فارغة. من الواضح أنه بسبب وجود حقيبة في الزاحف. ربما تكون الميزة مهمة جدًا. إذا علمنا أن الكلمات القريبة مثل "الجبن" و"الحليب" و"اللحم" - فهي بالتأكيد بقالة. ولن يقتصر النموذج على لغة واحدة فحسب، بل سيكون خاصًا بمجال واحد. عندما نزحف إلى متجر مواد البناء لن يكون هناك أي "جبن". أحب أن أحاول بناء نموذج قوي لاختلافات المجال واللغة. أريد تحليل المحلات التجارية هذا كل شيء.



#### img_seen


In [ ]:
df_clean[df_clean.y == 1]['img_seen'].mean()

In [ ]:
df_clean[df_clean.y == 0]['img_seen'].mean()


نعم، هناك صورة داخل كل بطاقة منتج



#### href_seen


In [ ]:
df_clean[df_clean.y == 1]['href_seen'].mean()

In [ ]:
df_clean[df_clean.y == 0]['href_seen'].mean()


و href داخل كل بطاقة منتج



## الجزء الخامس. هندسة الميزات ووصفها



تمت بعض معالجة الميزات أعلاه وحصلنا على الحجم والموضع


In [ ]:
df_clean.head(3)


تم تنفيذ بعض أجزاء FE في مرحلة الزاحف. ميزات مثل href_seen وimg_seen تعني أن بعض العناصر الفرعية تحتوي على img أو href بداخلها، ولكنها خارج الحدود. ربما يعني وجود 2-3-4 عناصر تنازلية أن الصورة قريبة، ولكن ليس على طول الطريق.
القيد الرئيسي الآن هو أنه في مجموعة البيانات لم يتم حفظ العلاقات بين العناصر. أولاً حصلت على معرف الصف، ولكن لا توجد إشارة إلى أن هذا المعرف هو أحد الوالدين في مكان ما. إصلاح هذا في التكرار التالي سيسمح لي بإنشاء المزيد من الميزات التابعة.
ولكن حتى الآن يمكننا إعادة إنشاء ترتيب علامات الوالدين.


In [ ]:
df_clean['parent_1'] = df_clean.parent_tag.apply(lambda x: x[-1])
df_clean['parent_2'] = df_clean.parent_tag.apply(lambda x: x[-2] if len(x) > 1 else '')
df_clean['parent_3'] = df_clean.parent_tag.apply(lambda x: x[-3] if len(x) > 2 else '')

In [ ]:
df_clean[df_clean.y == 1].sample(2)


باستخدام مرجع الأصل والمعرف، سأكون قادرًا على إضافة ميزات مثل <b>ankle_count</b> (إخوة الوالدين =))، <b>parent_size</b>، <b>parent_location</b>، <b>parent_class</b> وما إلى ذلك. لكن هذا غير ممكن الآن.


قد تكون الميزة الجيدة الأخرى المحتملة هي وجود عدد من الكائنات "المشابهة" على الصفحة. مماثلة من حيث أسماء فئات CSS أو من حيث الحجم والموقع. لكن ليس لدي أي إمكانية للقيام بذلك الآن نظرًا لعدم وجود ميزة "url" لذا لا يمكنني تنفيذها الآن.
قد تتساءل لماذا لم أقم بإصلاح خصائص الزاحف هذه. يعد خط الأنابيب السريع خاصية مهمة لبناء نظام تعلم الآلة. على الرغم من أنني لا أملك نتائج القياس، إلا أنني لن أكون عالقًا في التحسين اللانهائي للخطوات السابقة.



يمكننا استخدام element_classes_words، ولكن نظرًا لأنه قريب من العمل مع المفردات، فيجب أن نفعل ذلك فقط بعد تقسيم التحقق من الصحة.



## الجزء السادس. اختيار المقاييس


In [ ]:
df_clean.y.value_counts()


أولاً أريد أن أشير إلى أن وظيفة الخسارة والنتيجة المترية قد تكون مختلفة. 



#### متري



لدينا مشكلة في التصنيف وفئة غير متوازنة، لذا فإن استخدام أبسط دقة مقاييس ممكنة ليس جيدًا. الحالات الإيجابية الكاذبة والسلبية الكاذبة ليس لها أي أولوية. عندما يفتقد الزاحف الجديد بطاقات المنتج الصحيحة، سيتم فقدان بعض المعلومات. عندما يقوم الزاحف بتحديد بطاقة المنتج بشكل خاطئ في قاعدة البيانات سيتم تخزين المعلومات غير المرغوب فيها. في الواقع، سيتم إنشاء اسم المنتج وسعر المنتج وصورة المنتج ومعرفات فئة المنتج أعلى معرف بطاقة المنتج هذا. لذا فإن المعلومات المهملة ستسبب مشاكل على طول الخط. الآن أعتقد أن FP وFN لهما نفس الوزن.
لذلك ربما نرغب في التحقق من بعض المقاييس لفهم سلوك نموذجنا. ستكون ROC AUC وPR AUC ودرجة F1 ومصفوفة الارتباك. من المستحيل أن تتحسن في جميع المقاييس، لذا سأختار واحدًا بالضبط لاحقًا.



أنا أحب العلاقات العامة بالجامعة الأمريكية بالقاهرة أكثر من غيرها. PR اختصار لـ <b>Precision Recall</b>. إنهم يشبهون إلى حد ما ROC AUC.يرسم منحنى ROC المعدل الإيجابي الحقيقي (TPR) مقابل المعدل الإيجابي الكاذب (FPR).
يرسم منحنى العلاقات العامة الدقة مقابل الاستدعاء.
العلاقات العامة لا تأخذ في الاعتبار السلبيات الحقيقية (نظرًا لأن TN ليس أحد مكونات الدقة أو الاستدعاء). لذا، إذا كان هناك الكثير من السلبيات أكثر من الإيجابيات (وهي سمة من سمات مشكلة عدم التوازن الطبقي)، فيجب علينا استخدام العلاقات العامة.
http://www.chioka.in/differences-between-roc-auc-and-pr-auc/



#### الخسارة



الخيار الافتراضي للخسارة اللوغاريتمية ولكنه قد يكون خاطئًا. سوف أذكرك.


In [ ]:
Image(url='https://habrastorage.org/webt/p7/u6/cp/p7u6cppqgqauzlsenfrsfdymygw.png', width=900)


هذا هو <b>different</b> العناصر، تم وضع علامة على أحدها على أنه صحيح والآخر على أنه خطأ. إنه "div" داخل "li"، شكله دقيق تقريبًا، وموضعه دقيق تقريبًا. هل سيكون من العدل معاقبة نموذجنا الذي يخمن بطاقتي المنتج؟



## الجزء 7. اختيار النموذج



ليس لدينا الكثير من الميزات وأطنان من البيانات، لذا فمن المؤكد عدم استخدام الشبكات العصبية والتعزيز. البيانات بسيطة إلى حد ما لذا يجب أن يتطابق النموذج. من التحليل السابق رأينا أن العلامة الأصلية مع مجموعة الأشقاء_عدد يمكن أن تعني الكثير (إذا كان هدفنا "div" داخل الأشقاء "li" لا يعني شيئًا إذا كان div داخل الأشقاء div - يرتبط بالهدف)، فلا يمكن التقاط تفاعلات الميزات المعقدة هذه بواسطة النموذج الخطي. من المفترض أن تعمل الغابات العشوائية مثل السحر (سأحاول أيضًا إنشاء نموذج خطي).



## الجزء 8. المعالجة المسبقة للبيانات


In [ ]:
df_clean.head(1)

In [ ]:
df_clean.info()


لا يمكن استخدام "childs_tags". "النص" دائما NaN. "لقطة الشاشة" تستخدم فقط للتحقق البصري. لا يمكن استخدام "shop" كمقياس لأننا نستهدف المتاجر الجديدة (لكنني أحتاجه لفترة من الوقت). سيتم استخدام "element_classes_words" بعد قليل. تم إسقاط كافة الحقول الأخرى في وقت سابق. أنا أحمل y في X لفترة من الوقت، تحملني.


In [ ]:
features_to_train = ['depth', 'element_classes_count', 'href_seen', 'img_seen',
              'inner_elements', 'is_href', 'siblings_count', 'tag', 'loc_x',
             'loc_y', 'size_h', 'size_w', 'parent_1','parent_2', 'parent_3', 'shop', 'y']
categorical_columns = ['tag', 'parent_1','parent_2','parent_3']
continuous_columns = ['depth', 'element_classes_count','inner_elements','siblings_count',
                      'loc_x','loc_y', 'size_h', 'size_w']

In [ ]:
X = df_clean[features_to_train]
y = df_clean.y


بالنسبة إلى الترددات اللاسلكية، لا يتعين علينا إنشاء OHE، لذا سأقوم فقط بتحويل str إلى int.


In [ ]:
possible_tags_categories = X['tag'].astype('category')
possible_parent_categories = X['parent_1'].astype('category')
all_tags = list(possible_tags_categories.cat.categories) + list(possible_parent_categories.cat.categories)
num_to_category = {i:cat for i, cat in enumerate(all_tags)}
category_to_num = {cat:i for i, cat in num_to_category.items()}

In [ ]:
X['tag'] = X['tag'].apply(lambda x: category_to_num[x])
X['parent_1'] = X['parent_1'].apply(lambda x: category_to_num[x] if x else -1)
X['parent_2'] = X['parent_2'].apply(lambda x: category_to_num[x] if x else -1)
X['parent_3'] = X['parent_3'].apply(lambda x: category_to_num[x] if x else -1)

In [ ]:
X.head(1)

#### المعالجة المسبقة للنموذج الخطي


In [ ]:
X_linear = df_clean[['depth', 'element_classes_count', 'href_seen', 'img_seen',
              'inner_elements', 'is_href', 'siblings_count', 'tag', 'loc_x',
             'loc_y', 'size_h', 'size_w', 'parent_1','parent_2', 'parent_3', 'shop', 'y']]
y = df_clean.y


بالنسبة للنماذج الخطية، نحتاج إلى ميزات رقمية متدرجة وOHE للفئوية


In [ ]:
X_linear = pd.get_dummies(X_linear, columns=categorical_columns, drop_first=True,
                            prefix=categorical_columns, sparse=False)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_linear[continuous_columns] = scaler.fit_transform(X_linear[continuous_columns])

In [ ]:
X_linear.head(2)


الآن لدينا بيانات للنماذج الخطية والترددات اللاسلكية



## الجزء 9. التحقق من الصحة وتعديل المعلمات الفائقة للنموذج


In [ ]:
def print_scores(y_true, preds, verbose=True):
    f1 = f1_score(y_true, preds)
    roc_auc = roc_auc_score(y_true, preds)
    pr_auc = average_precision_score(y_true, preds)
    if verbose:
        print(f'F1 score is {f1}')
        print(f'ROC_AUC score is {roc_auc}')
        print(f'PR_AUC is {pr_auc}')
        print(f'confusion_matrix \n {confusion_matrix(y_true, preds)}')
    return f1, roc_auc, pr_auc


من المفترض أن يبدو النهج المعتاد لتقسيم البيانات بهذا الشكل. لن ننسى إسقاط المتغير المستهدف


In [ ]:
X_train,X_val,y_train,y_val = train_test_split(X.drop(['shop', 'y'],axis=1),y, random_state=42)
X_train.shape,X_val.shape,y_train.shape,y_val.shape


#### غابة عشوائية


In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
preds = rf.predict(X_val)
print_scores(y_val, preds);


يا إلهي، نحن مثاليون! في الواقع لا. الطريقة الوحيدة لإجراء اختبار معقول هي فصل جميع الأمثلة لمتجر واحد. لا يهم عدد الأمثلة التي نتركها 5% أو 50%. يجب أن نتعلم في المتاجر الموجودة لدينا ونستخدمها في البرية في متاجر أخرى لم نزحف إليها من قبل.
من الأفضل أن تفعل ذلك.



سأنفذ النتيجة المتقاطعة باليد


In [ ]:
diff_shops_df = [ (shop_name, shop_df) for shop_name, shop_df in X.groupby('shop')]
for shop_name, shop_df in diff_shops_df:
    y_cross_val = shop_df.y
    X_cross_val = shop_df.drop(['y', 'shop'],axis=1) # validation only 1 shop
    X_cross_train = X.drop(X_cross_val.index) # train all X without 1 shop
    y_cross_train = X_cross_train.y
    X_cross_train = X_cross_train.drop(['shop', 'y'],axis=1)
    
    rf = RandomForestClassifier(n_estimators=100,max_features='sqrt', criterion='entropy', min_samples_leaf=5,n_jobs=-1,)
    rf.fit(X_cross_train,y_cross_train)
    preds = rf.predict(X_cross_val)
    print(f'**********shop - {shop_name}*************')
    print(f'Train shape {X_cross_train.shape}')
    print(f'Val shape {X_cross_val.shape}')
    print_scores(y_cross_val, preds);


النتائج كارثة.



تقريبًا في كل مرة نحصل فيها على تنبؤات خاطئة بنسبة 100%، دعونا نشاهدها


In [ ]:
X_train_analize = X.drop(X[X.shop == 'Европа'].index)
X_val_analize = X[X.shop == 'Европа']
y_train_analize = X_train_analize.y
y_val_analize = X_val_analize.y
X_train_analize.drop(['shop','y'],axis=1, inplace=True)
X_val_analize.drop(['shop','y'],axis=1, inplace=True)

rf = RandomForestClassifier(n_estimators=100,max_features='sqrt', criterion='entropy', min_samples_leaf=5,n_jobs=-1,)
rf.fit(X_train_analize,y_train_analize)
preds = rf.predict(X_val_analize)
print_scores(y_val_analize, preds);

In [ ]:
X_val_analize[y_val_analize == True].head(5)

In [ ]:
X_val_analize[preds == True].head(5)


تحقق من الفهارس للتحقق من الصحة وtrue_y. ومن المثير للريبة أنهم يختلفون في 1.


In [ ]:
df_clean[df_clean.index.isin(X_val_analize[preds == True].index)].head(3)

In [ ]:
# screen for df_clean[df_clean.index.isin(X_val_analize[preds == True].index)].screenshot
Image(url='https://habrastorage.org/webt/x1/no/c5/x1noc54flobtv8lyf3ip66te05k.png', width=900)


نعم، العناصر الصحيحة، ولكن ظهرت بعض الالتباسات مع مقالة div-div-li-المتداخلة. كبشر، نرى التصنيف صحيحًا، لكن الآلة تفكر بشكل مختلف.
ومن خلال مصفوفة الارتباك يمكننا أن نرى عدد FN == 360 و FP == 369. لذلك ربما تم تصنيف 9 عناصر بشكل خاطئ. أعتقد أننا نتعامل مع بطاقة المنتج الصحيحة فقط مع تلك الموجودة في الكتالوج الرئيسي، ولكنها دائمًا تحتوي على بعض العروض الترويجية على الجانبين.



دعونا نتحقق من "المترو" أيضًا.


In [ ]:
X_train_analize = X.drop(X[X.shop == 'Метро'].index)
X_val_analize = X[X.shop == 'Метро']
y_train_analize = X_train_analize.y
y_val_analize = X_val_analize.y
X_train_analize.drop(['shop','y'],axis=1, inplace=True)
X_val_analize.drop(['shop','y'],axis=1, inplace=True)

rf = RandomForestClassifier(n_estimators=100,max_features='sqrt', criterion='entropy', min_samples_leaf=5,n_jobs=-1,)
rf.fit(X_train_analize,y_train_analize)
preds = rf.predict(X_val_analize)
print_scores(y_val_analize, preds);

In [ ]:
X_val_analize[y_val_analize == True].head(5)

In [ ]:
X_val_analize[preds == True].head(5)

In [ ]:
df_clean[df_clean.index.isin(X_val_analize[preds == True].index)].head(3)

In [ ]:
# screen for df_clean[df_clean.index.isin(X_val_analize[preds == True].index)].screenshot
Image(url='https://habrastorage.org/webt/to/4g/uw/to4guwglcb6epg6pf12bjehd69k.png', width=900)


نفس القصة مع بطاقات "Metro" الصحيحة، ولكن العناصر الخاطئة.
لكن مصفوفة الارتباك أسوأ FN == 520 FP == 209، نحن نفتقد الكثير.



جميع المحلات التجارية الأخرى تفعل أسوأ بكثير. كل التوقعات بالنسبة لهم كاذبة. من الصعب تحليلها.


In [ ]:
X_train_analize = X.drop(X[X.shop == 'Окей'].index)
X_val_analize = X[X.shop == 'Окей']
y_train_analize = X_train_analize.y
y_val_analize = X_val_analize.y
X_train_analize.drop(['shop','y'],axis=1, inplace=True)
X_val_analize.drop(['shop','y'],axis=1, inplace=True)

rf = RandomForestClassifier(n_estimators=100,max_features='sqrt', criterion='entropy', min_samples_leaf=5,n_jobs=-1,)
rf.fit(X_train_analize,y_train_analize)
preds = rf.predict(X_val_analize)
print_scores(y_val_analize, preds);

In [ ]:
# Extra module pip install treeinterpreter
from treeinterpreter import treeinterpreter as ti

In [ ]:
test_example = X_val_analize[y_val_analize == False].sample(1)
print(rf.predict_proba(test_example))

In [ ]:
test_example = X_val_analize[y_val_analize == True].sample(1)
print(rf.predict_proba(test_example))

في الاحتمالات، نرى الفرق بين الطبقة الصحيحة وغير الصحيحة. يختار تري الفئة الأكثر احتمالا، ولكن ربما يمكننا الاستفادة من نهج آخر


In [ ]:
prediction, bias, contributions = ti.predict(rf, test_example)
prediction, bias


الغابة متحيزة للغاية تجاه الافتراضات الخاطئة، دعنا نختار العناصر التي يكون احتمال أن يكون صحيحًا ليس > 0.5، ولكن > التحيز.


In [ ]:
preds_proba = rf.predict_proba(X_val_analize)
true_class_probs = preds_proba[:,1]
(true_class_probs > 0.023).sum()


1890 عنصر. الكثير لمتجرنا، ولكن من يهتم. دعونا نحلل المزيد.


In [ ]:
# screen for df_clean[df_clean.index.isin((true_class_probs > 0.023).index)].screenshot
Image(url='https://habrastorage.org/webt/yk/sk/t5/ykskt5lziqsgazssbkkk8vgm3w8.png', width=900)

In [ ]:
# screen for df_clean[df_clean.index.isin((true_class_probs > 0.023).index)].screenshot
Image(url='https://habrastorage.org/webt/yd/aj/kz/ydajkzlfdllxmdmus-v5vunfwcm.png', width=900)


إنها 778 عنصرًا حقيقيًا لـ "Okей" بينما نتوقع 1890 فوق التحيز. انظر عن كثب إلى صور العناصر الصحيحة، بعضها يبلغ ارتفاعه حوالي 330 بكسل، وبعضها يبلغ ارتفاعه حوالي 300 بكسل. إنها مشكلتنا المفضلة div في div. أفترض أن هناك تنبؤين لكل عنصر صحيح. إذن لدينا 778*2 = 1556 تنبؤات "صحيحة" و1890-1556=334 FP. بالطبع، يمكنني تعديل مستوى الثقة كما أريد، لكن مصطلح التحيز يعطيني إشارة واضحة إلى توزيع المتجر.
دعونا نعتمد التوقعات فوق مستوى التحيز.


In [ ]:
diff_shops_df = [ (shop_name, shop_df) for shop_name, shop_df in X.groupby('shop')]
f1s = list()
rocs = list()
prs = list()
for shop_name, shop_df in diff_shops_df:
    y_cross_val = shop_df.y
    X_cross_val = shop_df.drop(['y', 'shop'],axis=1) # validation only 1 shop
    X_cross_train = X.drop(X_cross_val.index) # train all X without 1 shop
    y_cross_train = X_cross_train.y
    X_cross_train = X_cross_train.drop(['shop', 'y'],axis=1)
    
    rf = RandomForestClassifier(n_estimators=100,max_features='sqrt', criterion='entropy', min_samples_leaf=5,n_jobs=-1,)
    rf.fit(X_cross_train,y_cross_train)
    _, bias, _ = ti.predict(rf,test_example) 
    bias_treshold = bias[0][1]
    preds_proba = rf.predict_proba(X_cross_val)
    preds = (preds_proba[:,1] > bias_treshold)
    print(f'**********shop - {shop_name}*************')
    f1, roc_auc, pr_auc = print_scores(y_cross_val, preds);
    f1s.append(f1)
    rocs.append(roc_auc)
    prs.append(pr_auc)
print(f'Mean f1 - {np.array(f1s).mean()}')
print(f'Mean ROC AUC - {np.array(rocs).mean()}')
print(f'Mean PR AUC - {np.array(prs).mean()}')


#### التصنيف الخطي


In [ ]:
diff_shops_df = [ (shop_name, shop_df) for shop_name, shop_df in X_linear.groupby('shop')]
f1s = list()
rocs = list()
prs = list()
for shop_name, shop_df in diff_shops_df:
    y_cross_val = shop_df.y
    X_cross_val = shop_df.drop(['y', 'shop'],axis=1) # validation only 1 shop
    X_cross_train = X_linear.drop(X_cross_val.index) # train all X without 1 shop
    y_cross_train = X_cross_train.y
    X_cross_train = X_cross_train.drop(['shop', 'y'],axis=1)
    
    sgd_l1 = SGDClassifier(loss="hinge", penalty="l2", max_iter=5)
    sgd_l1.fit(X_cross_train,y_cross_train)
    preds = sgd_l1.predict(X_cross_val)
    print(f'**********shop - {shop_name}*************')
    f1, roc_auc, pr_auc = print_scores(y_cross_val, preds);
    f1s.append(f1)
    rocs.append(roc_auc)
    prs.append(pr_auc)
print(f'Mean f1 - {np.array(f1s).mean()}')
print(f'Mean ROC AUC - {np.array(rocs).mean()}')
print(f'Mean PR AUC - {np.array(prs).mean()}')


يبدو أكثر واعدة من خارج منطقة الجزاء. يتيح ضبط المعلمات الفائقة.
أحتاج إلى طريقة خاصة لتنفيذ السيرة الذاتية، في بيانات التدريب والاختبار التي من المفترض أن تكون متاجر مختلفة. يوفر GroupShuffleSplit الوظائف المطلوبة. دعونا نرى كيف يعمل.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
y_tmp = X.y
X_tmp = X.drop(['y', 'shop'], axis=1)
g = GroupShuffleSplit(n_splits=5,random_state=56)
itr = g.split(X, y=X.y, groups=X.shop.values,)
for tr, tst in itr:
    print(tr.shape, tst.shape)


هناك انقسامات متكررة، ولم يتم تقييم جميع المتاجر الخمسة كبيانات التحقق من الصحة. لا يمكنني التضحية حتى بمتجر واحد عند التحقق من صحته، فهناك عدد قليل جدًا منهم.
تنفيذ سريع جدًا وقذر لـ GridSearch باستخدام GroupShuffleSplit.


In [ ]:
diff_shops_df = [ (shop_name, shop_df) for shop_name, shop_df in X_linear.groupby('shop')]
f1s = list()
rocs = list()
prs = list()
best_scores = dict()
losses = ['hinge', 'log', 'perceptron']
penalties = ['l2', 'l1', 'elasticnet']
max_iter = [5,10,20,40]
for l in losses:
    for p in penalties:
        for it in max_iter:
            for shop_name, shop_df in diff_shops_df:
                y_cross_val = shop_df.y
                X_cross_val = shop_df.drop(['y', 'shop'],axis=1) # validation only 1 shop
                X_cross_train = X_linear.drop(X_cross_val.index) # train all X without 1 shop
                y_cross_train = X_cross_train.y
                X_cross_train = X_cross_train.drop(['shop', 'y'],axis=1)

                sgd_l1 = SGDClassifier(loss=l, penalty=p, max_iter=it)
                sgd_l1.fit(X_cross_train,y_cross_train)
                preds = sgd_l1.predict(X_cross_val)
                f1, roc_auc, pr_auc = print_scores(y_cross_val, preds, verbose=False);
                f1s.append(f1)
                rocs.append(roc_auc)
                prs.append(pr_auc)
            f1_mean = np.array(f1s).mean()
            roc_mean = np.array(rocs).mean()
            pr_mean = np.array(prs).mean()
            best_scores[l+p+str(it)] = [f1_mean, roc_mean, pr_mean]

In [ ]:
print(f'best by f1 {max(best_scores.items(), key=(lambda key: key[1][0]))}') 
print(f'best by roc {max(best_scores.items(), key=(lambda key: key[1][1]))}') 
print(f'best by pr {max(best_scores.items(), key=(lambda key: key[1][2]))}') 


أفضل المعلمات الفائقة لدينا هي logloss، والتنظيم المرن، و40 تكرارًا. 40 كان الحد الأقصى للرقم في بحثنا على الشبكة، لذا ربما يمكننا زيادته.


In [ ]:
X_train_analize = X_linear.drop(X_linear[X_linear.shop == 'Окей'].index)
X_val_analize = X_linear[X_linear.shop == 'Окей']
y_train_analize = X_train_analize.y
y_val_analize = X_val_analize.y
X_train_analize.drop(['shop','y'],axis=1, inplace=True)
X_val_analize.drop(['shop','y'],axis=1, inplace=True)

sgd_l1 = SGDClassifier(loss='log', penalty='elasticnet', max_iter=40)
sgd_l1.fit(X_train_analize,y_train_analize)
preds = sgd_l1.predict(X_val_analize)
print_scores(y_val_analize, preds);

In [ ]:
X_train_analize = X_linear.drop(X_linear[X_linear.shop == 'Метро'].index)
X_val_analize = X_linear[X_linear.shop == 'Метро']
y_train_analize = X_train_analize.y
y_val_analize = X_val_analize.y
X_train_analize.drop(['shop','y'],axis=1, inplace=True)
X_val_analize.drop(['shop','y'],axis=1, inplace=True)

sgd_l1 = SGDClassifier(loss='log', penalty='elasticnet', max_iter=40)
sgd_l1.fit(X_train_analize,y_train_analize)
preds = sgd_l1.predict(X_val_analize)
print_scores(y_val_analize, preds);


لا يزال "Metro" يفشل بشدة، ولكن كعادة دعونا نتحقق مما يتوقعه.


In [ ]:
X_val_analize[preds == True].head(5)

In [ ]:
X_val_analize[y_val_analize == True].head(5)

بعد كل التحولات، من الغامض استنتاج نمط التنبؤ الخاطئ. لحسن الحظ لدينا لقطات لدينا.


In [ ]:
# screen for df_clean[df_clean.index.isin((preds == True).index)].screenshot
Image(url='https://habrastorage.org/webt/5j/jb/hg/5jjbhgbqzophvn8pdl_2wxicpky.png', width=900)


حصلنا على نموذجين RF وLR مع النتائج:
|    - | الترددات اللاسلكية | إل آر |
|-------|------|-----|
|   f1 | 0.449|0.255|
|ROC الجامعة الأمريكية | 0.907|0.632|
| العلاقات العامة الجامعة الأمريكية بالقاهرة| 0.280|0.240|



الشيء الجيد هو أن نماذجنا تعمل بشكل جيد، والشيء السيئ هو أننا لا نستطيع فهمها من خلال الدرجات المترية.



### الجزء العاشر. رسم منحنيات التدريب والتحقق من الصحة



في الوقت الحالي سأختار PR AUC كمقياس.



#### معلمات الترددات اللاسلكية n_estimators


In [ ]:
X_train_analize = X.drop(X[X.shop == 'Метро'].index)
X_val_analize = X[X.shop == 'Метро']
y_train_analize = X_train_analize.y
y_val_analize = X_val_analize.y
X_train_analize.drop(['shop','y'],axis=1, inplace=True)
X_val_analize.drop(['shop','y'],axis=1, inplace=True)

trees_num = np.linspace(1,500, dtype=int)
pr_score_val = list()
pr_score_train = list()
for tr_num in trees_num:
    rf = RandomForestClassifier(n_estimators=tr_num,max_features='sqrt', criterion='entropy', min_samples_leaf=5,n_jobs=-1,)
    rf.fit(X_train_analize,y_train_analize)
    _, bias, _ = ti.predict(rf,test_example) 
    bias_treshold = bias[0][1]
    preds_proba_val = rf.predict_proba(X_val_analize)
    preds_val = (preds_proba_val[:,1] > bias_treshold)
    _,_, pr_auc_val = print_scores(preds_val, y_val_analize, verbose=False);
    
    preds_proba_train = rf.predict_proba(X_train_analize)
    preds_train = (preds_proba_train[:,1] > bias_treshold)
    _,_, pr_auc_train = print_scores(preds_train, y_train_analize, verbose=False);
    
    pr_score_train.append(pr_auc_train)
    pr_score_val.append(pr_auc_val)

In [ ]:
fig, ax = plt.subplots()
ax.plot(trees_num,pr_score_train, label='train')
ax.plot(trees_num,pr_score_val, label='valid')
ax.legend()
ax.set(xlabel='N_estimators', ylabel='PR AUC', );


نرى أن الغابة مكتظة بشكل كبير. يمكننا تقليل التجهيز الزائد عن طريق min_samples_leaf.



#### معلمة التردد اللاسلكي min_samples_leaf


In [ ]:
X_train_analize = X.drop(X[X.shop == 'Метро'].index)
X_val_analize = X[X.shop == 'Метро']
y_train_analize = X_train_analize.y
y_val_analize = X_val_analize.y
X_train_analize.drop(['shop','y'],axis=1, inplace=True)
X_val_analize.drop(['shop','y'],axis=1, inplace=True)

min_leaf_list = np.linspace(1,100, dtype=int)
pr_score_val = list()
pr_score_train = list()
for leaf_num in min_leaf_list:
    rf = RandomForestClassifier(n_estimators=500,max_features='sqrt', criterion='entropy', min_samples_leaf=leaf_num,n_jobs=-1,)
    rf.fit(X_train_analize,y_train_analize)
    _, bias, _ = ti.predict(rf,test_example) 
    bias_treshold = bias[0][1]
    preds_proba_val = rf.predict_proba(X_val_analize)
    preds_val = (preds_proba_val[:,1] > bias_treshold)
    _,_, pr_auc_val = print_scores(preds_val, y_val_analize, verbose=False);
    
    preds_proba_train = rf.predict_proba(X_train_analize)
    preds_train = (preds_proba_train[:,1] > bias_treshold)
    _,_, pr_auc_train = print_scores(preds_train, y_train_analize, verbose=False);
    
    pr_score_train.append(pr_auc_train)
    pr_score_val.append(pr_auc_val)

In [ ]:
fig, ax = plt.subplots()
ax.plot(min_leaf_list,pr_score_train, label='train')
ax.plot(min_leaf_list,pr_score_val, label='valid')
ax.legend()
ax.set(xlabel='min_samples_leaf', ylabel='PR AUC', );


لذا فإن min_samples_leaf لا يساعد كثيرًا.



ربما نحتاج إلى المزيد من البيانات؟


In [ ]:
y_tmp = X.y
X_tmp = X.drop(['y', 'shop'], axis=1)
groups = X.shop.values

def plot_with_err(x, data, **kwargs):
    mu, std = data.mean(1), data.std(1)
    lines = plt.plot(x, mu, '-', **kwargs)
    plt.fill_between(x, mu - std, mu + std, edgecolor='none',
                     facecolor=lines[0].get_color(), alpha=0.2)

def plot_learning_curve(classifier = 'rf'):
    if classifier == 'rf':
        classif = RandomForestClassifier(n_estimators=100,max_features='sqrt', criterion='entropy', min_samples_leaf=25,n_jobs=-1,)
    else:
        classif = SGDClassifier(loss='log', penalty='elasticnet', max_iter=40)
    N_train, val_train, val_test = learning_curve(classif, X_tmp, y_tmp,
                                                  cv=5, groups=groups,
                                                  shuffle=True, scoring='average_precision'
                                          )
    plot_with_err(N_train, val_train, label='training scores')
    plot_with_err(N_train, val_test, label='validation scores')
    plt.xlabel('Training Set Size'); plt.ylabel('PR AUC')
    plt.legend()
    plt.grid(True);

In [ ]:
plot_learning_curve(classifier='rf')

In [ ]:
plot_learning_curve(classifier='lr')


قد تكون هذه الرسوم البيانية مضللة. لا أستطيع تفسيرها حقًا، فالتباين كبير جدًا.
هناك حوالي 100 ألف من الأمثلة التدريبية ولكن في الواقع، لدينا 5 متاجر فقط. كل واحد منهم لديه نوع واحد فقط من عربة المنتجات. لذا فهي <b>not</b> 100 ألف مثال، إنها حوالي 5 أمثلة إلى حد ما. البيانات الإضافية ستكون مفيدة جدًا جدًا.



## الجزء 11. التنبؤ بالعينات الاختبارية أو المحتجزة



كاختبار، سنقوم بالزحف إلى موقع آخر - أوشان. لن يكون هناك أي فحص للميزات. سنقوم فقط بفحص صور بطاقة المنتج للتأكد من أن الزاحف يعمل بشكل صحيح.


In [ ]:
df_test = pd.read_csv('test_sites_markup.csv')

In [ ]:
# screen for df_test[(df_test.y == 1)].screenshot
Image(url='https://habrastorage.org/webt/sv/-m/ac/sv-macheei3wqmgkpdojbuky734.png', width=900)


#### إعداد البيانات


In [ ]:
df_test[['loc_x', 'loc_y']] = df_test['location'].str.replace("'", '"').apply(json.loads).apply(pd.Series)
df_test[['size_h', 'size_w']] = df_test['size'].str.replace("'", '"').apply(json.loads).apply(pd.Series)
df_test.size_h = df_test.size_h.astype(int)
df_test.size_w = df_test.size_w.astype(int)

In [ ]:
df_test['parent_tag'] = df_test.parent_tag.str.split('_')
df_test['childs_tags'] = df_test.childs_tags.apply(lambda x: x.replace('[', '').replace(']','').strip())

In [ ]:
df_clean_test = df_test.copy()
df_clean_test.drop(['location', 'size', 'Unnamed: 0'], axis=1, inplace=True)

In [ ]:
df_clean_test['parent_1'] = df_clean_test.parent_tag.apply(lambda x: x[-1])
df_clean_test['parent_2'] = df_clean_test.parent_tag.apply(lambda x: x[-2] if len(x) > 1 else '')
df_clean_test['parent_3'] = df_clean_test.parent_tag.apply(lambda x: x[-3] if len(x) > 2 else '')

In [ ]:
X_test = df_clean_test[features_to_train]

In [ ]:
X_test['tag'] = X_test['tag'].apply(lambda x: category_to_num[x] if x in category_to_num else -1)
X_test['parent_1'] = X_test['parent_1'].apply(lambda x: category_to_num[x] if (x) and (x in category_to_num) else -1)
X_test['parent_2'] = X_test['parent_2'].apply(lambda x: category_to_num[x] if (x) and (x in category_to_num) else -1)
X_test['parent_3'] = X_test['parent_3'].apply(lambda x: category_to_num[x] if (x) and (x in category_to_num) else -1)


#### التنبؤ


In [ ]:
X_train = X
y_train = X_train.y
X_train = X_train.drop(['shop', 'y'],axis=1)

y_test = X_test.y
X_test = X_test.drop(['shop', 'y'],axis=1)

test_example = X_test.head(1)
rf = RandomForestClassifier(n_estimators=400,max_features='sqrt', criterion='entropy', min_samples_leaf=25,n_jobs=-1,)
rf.fit(X_train,y_train)
_, bias, _ = ti.predict(rf,test_example) 
bias_treshold = bias[0][1]
preds_proba = rf.predict_proba(X_test)
preds = (preds_proba[:,1] > bias_treshold)
f1, roc_auc, pr_auc = print_scores(y_test, preds);

In [ ]:
# screen for df_test[df_test.index.isin((preds == True).index)].screenshot
Image(url='https://habrastorage.org/webt/_5/_n/uk/_5_nuk1iphmid6sfplw7hmcph1c.png', width=900)


حلو



# الجزء 11. الاستنتاجات



لقد جربنا ML على تخمين ترميز HTML وقد نجح الأمر.
وفي مسار تحليلنا، فهمنا طرقًا لتحسين جمع البيانات. ما هي ميزات البيانات التي نفتقر إليها. الميزات التي تم إنشاؤها بشكل خاطئ.في البداية، لم نتمكن من تحديد ما هو الأهم من الأخطاء الإيجابية الكاذبة أو الأخطاء السلبية الكاذبة. بعد رحلتنا، فهمنا أن FN أكثر أهمية وأنه يتعين علينا التعامل مع FP. وهذا يعني أننا سنقوم ببناء خط أنابيب البيانات الخاص بنا مع هذا المطلب - أن يكون مستقرًا بالنسبة إلى FP.
تبين أن مجموعة التدريب 100 ألف عبارة عن مجموعة قطار مكونة من 5 أمثلة فقط. نظرًا لأن البطاقة الصحيحة متطابقة تقريبًا في المتجر، ولا يهم إذا كان لدينا بطاقة بقيمة 10 آلاف متجر، فمن الأفضل أن يكون لدينا بطاقة واحدة لمتجر يضم 10 آلاف متجر (لكنه مستحيل).
لم تنجح مقاييسنا (باستثناء مصفوفة الارتباك)، لأن بيانات القطار لدينا تحتوي على أمثلة تم تصنيفها على أنها خاطئة، ولكنها في الواقع مطابقة للأمثلة الصحيحة (مشكلة div-div-ul). إنها فرصة جيدة لتوضيح وظيفة الخسارة المخصصة التي تعاقب بهدوء عندما نكون أقرب إلى العنصر المستهدف. ربما يكون من المفيد إعادة تنسيق المشكلة إلى الانحدار، حتى نجد "المسافة" إلى العنصر الصحيح.
في اختيار النموذج، فاز RF، ولكن من السابق لأوانه الحديث عن ذلك لأن لدينا مشاكل في المقاييس ومجموعة البيانات الخاصة بنا.